# 3. Warmup phase retrieval only
Independent HAPRE → ER warmup on your corrected holograms. There is **no physical projection** and no joint outer loop. The red `supportmask.png` must match the centered object grid produced by notebook 2; red means allowed support, including object and reference holes.

The solver receives the already prepared grid with crop=0 and binning=1. This keeps the saved warmup fields on the same grid as their holograms. The largest connected support component is the default material region; inspect its overlay. All exact inputs, fields, diagnostics, and mode geometry are saved together.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
if HERE.name != "permanent_magnets":
    HERE = HERE / "permanent_magnets"
REPO = HERE.parent
sys.path.insert(0, str(REPO))
from permanent_magnets import workflow

from library import phase_retrieval_universal as universal

In [ ]:
PROCESSED = Path("/path/to/processed")  # Edit: same output folder in all four notebooks.
INPUT = PROCESSED / "02_holograms.h5"  # HDF5 input from the preceding notebook.
OUTPUT = PROCESSED / "03_warmup.h5"  # HDF5 file written by this notebook; overwritten on rerun.
SUPPORT_PNG = HERE / "masks" / "supportmask.png"  # Red = allowed object/support pixels.

# Prepared hologram grid (keep this geometry for the saved handoff)
CROP = 0  # Pixels removed from each edge; keep 0 in retrieval to preserve the saved grid.
BINNING = 1  # Detector binning factor; keep 1 for the prepared grid.
ROI = None  # Optional retrieval region; keep None to use the prepared grid.
HOLOGRAM_INTENSITY_CUTOFF_VMIN = -1  # -1 disables percentile background subtraction.

# Initialization
STARTIMAGE_SCALE_FIT = "through_origin"  # Scale initial intensity without fitting an offset.
MODE_INITIALIZATION = "support_fft"  # Initialize Fourier fields from the mode supports.

# Modes and coherence
MODES = [1, 2]  # [1] for a single mode; 2 enlarges the second support.
PARTIAL_COHERENCE = False  # Enable partial-coherence updates; False uses coherent retrieval.
FFT_WORKERS = 1  # FFT threads used for each observation update.

# Warmup schedule
WARMUP_MODE = ["HAPRE", "ER"]  # Retrieval algorithm for each warmup stage.
WARMUP_ITERATIONS = [320, 50]  # HAPRE, ER.
WARMUP_BETA_MODE = ["arctan", "const"]  # Feedback schedule for each warmup stage.
WARMUP_ALPHA_ZERO = .5  # Initial TV regularization step size during warmup.
WARMUP_TV_FREQ = 10  # Apply TV regularization every this many warmup iterations.
WARMUP_START_FROM_FIRST = False  # Seed later observations from the first warmup if True.

# Stage boundary: warmup only, no physical outer rounds
PROJECTION_MODEL = "none"  # Physical factorization model; "none" disables physical fitting.
OUTER_ITERATIONS = 0  # Number of joint rounds; keep 0 for warmup only.



In [ ]:
with h5py.File(INPUT, "r") as h:
    holograms = h["holograms"][()]
    mask_pixel = h["mask_pixel"][()].astype(bool)
    scan_indices = h["scan_indices"][()]
    energies = h["energy_eV"][()]
    polarizations = h["polarization"][()]
    state_labels = h["state_labels"].asstr()[()].tolist()
support = workflow.red_mask(SUPPORT_PNG, holograms.shape[-2:])
material = universal.largest_support_component(support).astype(float)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(support, cmap="gray"); axes[0].set_title("Painted support (object + references)")
axes[1].imshow(material, cmap="gray"); axes[1].set_title("Largest component: material region")
fth_preview = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(np.where(mask_pixel[0], 0, holograms[0]))))
axes[2].imshow(np.log1p(abs(fth_preview)))
axes[2].contour(support, levels=[.5], colors="red", linewidths=.5)
axes[2].set_title("FTH preview with support overlay")
plt.show()
# The library requires a recipe dictionary; editable controls stay as variables above.
recipe = universal.default_universal_phase_retrieval_recipe()
recipe.update(
    crop=CROP,
    binning=BINNING,
    roi=ROI,
    hologram_intensity_cutoff_vmin=HOLOGRAM_INTENSITY_CUTOFF_VMIN,
    startimage_scale_fit=STARTIMAGE_SCALE_FIT,
    mode_initialization=MODE_INITIALIZATION,
    modes=MODES,
    partial_coherence=PARTIAL_COHERENCE,
    warmup_mode=WARMUP_MODE,
    warmup_Nit=WARMUP_ITERATIONS,
    warmup_beta_mode=WARMUP_BETA_MODE,
    warmup_alpha_zero=WARMUP_ALPHA_ZERO,
    warmup_TV_freq=WARMUP_TV_FREQ,
    warmup_start_from_first=WARMUP_START_FROM_FIRST,
    fft_workers=FFT_WORKERS,
    projection_model=PROJECTION_MODEL,
    outer_iterations=OUTER_ITERATIONS,
)
_, warmup, components, bsmasks, errors = universal.universal_phase_retrieval_algorithm(
    holograms, mask_pixel, support, state_labels=state_labels,
    energy_labels=energies, polarization_coefficients=polarizations,
    illumination_labels=["beam0"]*len(state_labels), universal_recipe=recipe,
    phase_retrieval_kernel=universal.multimode_phase_retrieval_kernel,
)
assert warmup.shape[-2:] == holograms.shape[-2:]

In [ ]:
# Compute centered objects and measured-pixel residuals for saving.
mode_fields = warmup[:, None] if warmup.ndim == 3 else warmup
objects = np.stack([universal.fourier_field_to_display_object(mode_fields[:, i])
                    for i in range(mode_fields.shape[1])], axis=1)
predicted_intensity = np.sum(abs(mode_fields)**2, axis=1)
valid = ~np.asarray(bsmasks, bool) & np.isfinite(holograms)
intensity_residual = np.sqrt(
    np.sum(np.where(valid, (predicted_intensity-holograms)**2, 0), axis=(1, 2))
    / np.maximum(np.sum(np.where(valid, holograms**2, 0), axis=(1, 2)), 1e-30))
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with h5py.File(OUTPUT, "w") as h:
    data_mode, xmcd_pairs = workflow.pairs_from_state_labels(state_labels, polarizations, energies)
    h.attrs["data_mode"] = data_mode
    h.attrs["reference_index"] = 0  # Plot reference only; unrelated to physical grouping.
    h.create_dataset("xmcd_pairs", data=xmcd_pairs)
    h.attrs["source_file"] = str(INPUT.resolve())
    h.attrs["support_png"] = str(SUPPORT_PNG.resolve())
    h.attrs["crop"] = 0
    h.attrs["binning"] = 1
    h.attrs["partial_coherence"] = False
    for name, array in [("warmup_fields", warmup), ("objects", objects),
                        ("holograms", holograms), ("mask_pixel", mask_pixel),
                        ("warmup_bsmasks", bsmasks), ("supportmask", support),
                        ("material_thickness", material), ("modes", MODES),
                        ("warmup_iterations", WARMUP_ITERATIONS), ("scan_indices", scan_indices),
                        ("energy_eV", energies), ("polarization", polarizations),
                        ("intensity_residual", intensity_residual)]:
        h.create_dataset(name, data=array, compression="gzip")
    h.create_dataset("state_labels", data=state_labels, dtype=h5py.string_dtype())
    workflow.save_tree(h.create_group("components"), components)
    workflow.save_tree(h.create_group("errors"), errors)
    workflow.save_tree(h.create_group("recipe"), recipe)
print("Saved", OUTPUT)

## Plot XMCD or reference differences

XMCD: subtract negative polarization from positive polarization for each state. Single polarization: subtract the observation selected by `REFERENCE_INDEX` from every observation. Plot magnitude and real part of the complex difference.


In [ ]:
# Reload results; differences are formed from complex reconstructed objects.
RESULT_FILE = OUTPUT  # HDF5 result to inspect.
REFERENCE_INDEX = 0  # Plotting only: single-polarization reference observation.
DISPLAY_ROI = None  # Optional (row_start, row_stop, column_start, column_stop), object pixels.
ALIGN_DIFFERENCE_PHASE = True  # Match global phases in material-free reference holes before subtraction.
with h5py.File(RESULT_FILE, "r") as h:
    saved_fields = h["warmup_fields"][()]
    energies = h["energy_eV"][()]
    polarizations = h["polarization"][()]
    labels = h["state_labels"].asstr()[()].tolist()
    support = h["supportmask"][()].astype(bool)
    material = h["material_thickness"][()]
    data_mode = h.attrs["data_mode"]
    reference_index = REFERENCE_INDEX
    xmcd_pairs = h["xmcd_pairs"][()]
    residuals = h["intensity_residual"][()]
    propagation_um = float(h.attrs.get("focus_prop_um", 0.))
    phase_rad = float(h.attrs.get("focus_phase_rad", 0.))
    focus_setup = None
    if "focus_pixel_size_m" in h.attrs:
        focus_setup = {"px_size": h.attrs["focus_pixel_size_m"], "ccd_dist": h.attrs["focus_detector_distance_m"]}
pairs = workflow.difference_pairs(data_mode, xmcd_pairs, reference_index, polarizations, energies)
reference_mask = support & (material == 0) if ALIGN_DIFFERENCE_PHASE else None
saved_objects = workflow.focused_objects(saved_fields, energies, propagation_um, phase_rad, focus_setup)
differences = workflow.object_differences(saved_objects, pairs, reference_mask)
view = np.s_[:, :] if DISPLAY_ROI is None else np.s_[DISPLAY_ROI[0]:DISPLAY_ROI[1], DISPLAY_ROI[2]:DISPLAY_ROI[3]]
difference_labels = [f"State {i}: + minus -" if data_mode == "xmcd" else f"{labels[a]} minus reference {b}" for i, (a, b) in enumerate(pairs)]
for mode in range(differences.shape[1]):
    selected = differences[:, mode, view[0], view[1]]
    limit = max(float(np.percentile(abs(selected), 99.5)), 1e-12)
    fig, axes = plt.subplots(len(pairs), 2, figsize=(10, 3.5*len(pairs)), squeeze=False)
    for i, title in enumerate(difference_labels):
        absolute = axes[i, 0].imshow(abs(selected[i]), vmin=0, vmax=limit)
        real = axes[i, 1].imshow(selected[i].real, cmap="RdBu_r", vmin=-limit, vmax=limit)
        axes[i, 0].set_title(f"{title}: absolute difference")
        axes[i, 1].set_title(f"{title}: real difference")
        fig.colorbar(absolute, ax=axes[i, 0], shrink=.75); fig.colorbar(real, ax=axes[i, 1], shrink=.75)
    fig.suptitle(f"Warmup differences, mode {mode+1}"); fig.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(residuals, "o-"); ax.set(xlabel="Observation", ylabel="Relative intensity RMS", title="Measured-pixel agreement")
plt.show()


## Optional interactive focus on one set

Enable the sliders after entering measured detector geometry. Save the chosen focus, then rerun the difference plots. The physical notebook reads this focus from the warmup HDF5. Stored warmup fields remain on the detector grid.


In [ ]:
# Optional: tune focus on ONE difference, then apply the same focus to all sets.
ENABLE_FOCUS_SLIDERS = False  # True displays interactive propagation and phase sliders.
FOCUS_SET_INDEX = 0  # XMCD state or observation/reference difference used for tuning.
FOCUS_MODE_INDEX = 0  # Zero-based reconstruction mode to preview.
DETECTOR_PIXEL_SIZE_M = None  # Enter measured detector pixel size in metres before focusing.
DETECTOR_DISTANCE_M = None  # Enter measured sample-to-detector distance in metres.
FOCUS_RANGE_UM = 20.0  # Symmetric range of the propagation slider, in micrometres.
if ENABLE_FOCUS_SLIDERS:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    if DETECTOR_PIXEL_SIZE_M is None or DETECTOR_DISTANCE_M is None:
        raise ValueError("Enter detector pixel size and distance before enabling focus sliders")
    assert 0 <= FOCUS_SET_INDEX < len(pairs) and 0 <= FOCUS_MODE_INDEX < saved_objects.shape[1]
    preview_setup = {"px_size": DETECTOR_PIXEL_SIZE_M, "ccd_dist": DETECTOR_DISTANCE_M}
    propagation_slider = widgets.FloatSlider(value=propagation_um, min=-FOCUS_RANGE_UM, max=FOCUS_RANGE_UM, step=.1, description="Focus (um)", continuous_update=False)
    phase_slider = widgets.FloatSlider(value=phase_rad, min=-np.pi, max=np.pi, step=.01, description="Phase (rad)", continuous_update=False)
    preview_output = widgets.Output()
    save_button = widgets.Button(description="Save focus for all sets")
    def preview_focus(change=None):
        selected_pair = pairs[FOCUS_SET_INDEX]
        objects = workflow.focused_objects(saved_fields[selected_pair], energies[selected_pair], propagation_slider.value, phase_slider.value, preview_setup)
        difference = workflow.object_differences(objects, [[0, 1]], reference_mask)[0, FOCUS_MODE_INDEX][view]
        with preview_output:
            clear_output(wait=True)
            fig, axes = plt.subplots(1, 2, figsize=(10, 4))
            limit = max(float(np.percentile(abs(difference), 99.5)), 1e-12)
            axes[0].imshow(abs(difference), vmin=0, vmax=limit); axes[0].set_title("Absolute difference")
            axes[1].imshow(difference.real, cmap="RdBu_r", vmin=-limit, vmax=limit); axes[1].set_title("Real difference")
            plt.show(); plt.close(fig)
    def save_focus(button):
        with h5py.File(RESULT_FILE, "a") as h:
            h.attrs["focus_prop_um"] = propagation_slider.value
            h.attrs["focus_phase_rad"] = phase_slider.value
            h.attrs["focus_pixel_size_m"] = DETECTOR_PIXEL_SIZE_M
            h.attrs["focus_detector_distance_m"] = DETECTOR_DISTANCE_M
        with preview_output:
            print("Focus saved. Rerun the difference plot cell to apply it to every set; notebook 4 will use this focus for fitting.")
    propagation_slider.observe(preview_focus, names="value")
    phase_slider.observe(preview_focus, names="value")
    save_button.on_click(save_focus)
    display(widgets.VBox([propagation_slider, phase_slider, save_button, preview_output]))
    preview_focus()
